# Qwen state capture + choice over the full candidate set (Kaggle T4, internet ON)
Exploratory (no gate). Input: `rows.jsonl` from the builder notebook (attach that notebook's saved output as an input, or run in the same session).
Qwen/Qwen3-4B-Thinking-2507 is downloaded here at revision 768f209d9ea81521153ed38c47d515654e938aea and never trained.
One forward pass per decision over a pool prompt (state + lettered options incl. hard negatives and NONE). Saved per decision:
the hidden state at depths 18/24/30 for the query (last token) and for each option (its last token), plus Qwen's next-letter probabilities over the options (no generation, no thinking).
This is a NEW capture protocol on T4 (`pool_v1_kaggle`); it is not the registered A5-v3 capture, so compare only within this run.

In [ ]:
import os, sys, json, glob, hashlib, random, subprocess, time, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "transformers>=4.51"], check=True)
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
assert torch.cuda.is_available(), "enable a GPU (T4) in the notebook settings"
print(torch.cuda.get_device_name(0), torch.__version__)
REV = "768f209d9ea81521153ed38c47d515654e938aea"
CFG = dict(depths=[18, 24, 30], max_rows=None, shard=500, max_tokens=3500, state_chars=5000, seed=20261004, splits=None)  # splits=None means all
OUT = "/kaggle/working/head_states"; os.makedirs(OUT, exist_ok=True)
# Resume from a previously uploaded partial-states dataset
_prev = "/kaggle/input/laya-head-states-partial/head_states"
if os.path.exists(_prev):
    for _f in sorted(glob.glob(_prev + "/shard-*.pt")):
        _dst = OUT + "/" + os.path.basename(_f)
        if not os.path.exists(_dst): shutil.copy2(_f, _dst)
    print("resumed", len(glob.glob(OUT + "/shard-*.pt")), "shards from laya-head-states-partial")
cand = glob.glob("/kaggle/input/laya-head-rows-v1/rows.jsonl") + glob.glob("/kaggle/input/**/head_rows/rows.jsonl", recursive=True) + glob.glob("/kaggle/working/head_rows/rows.jsonl")
assert cand, "rows.jsonl not found: attach the builder notebook output as an input"
ROWS = cand[0]; print("rows file:", ROWS)
MOUNT = "/kaggle/input/qwen3-4b-thinking-2507"
if os.path.exists(MOUNT + "/config.json"):
    path, QWEN_SOURCE = MOUNT, "kaggle_mount_revision_unverified"   # a mounted copy carries no revision; compare file hashes with the HF manifest before trusting it
else:
    path, QWEN_SOURCE = snapshot_download("Qwen/Qwen3-4B-Thinking-2507", revision=REV), "hf_pinned"
    assert REV in path, path
print("Qwen from", QWEN_SOURCE, path)
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map="cuda", attn_implementation="sdpa").eval()
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
LET_IDS = [tok.encode(l, add_special_tokens=False)[0] for l in LETTERS]
print("hidden", model.config.hidden_size, "layers", model.config.num_hidden_layers)

In [ ]:
rows = [json.loads(l) for l in open(ROWS, encoding="utf-8")]
if CFG["splits"]: rows = [r for r in rows if r["split"] in CFG["splits"]]
if CFG["max_rows"]: rows = rows[:CFG["max_rows"]]
print(len(rows), "decisions")

def encode_pool(r):
    rng = random.Random(CFG["seed"] + int(hashlib.sha256(r["decision_id"].encode()).hexdigest()[:8], 16))
    order = list(range(len(r["candidates"]))); rng.shuffle(order)
    ids, ends = tok.encode("Task and recent history:\n" + r["state_text"][-CFG["state_chars"]:] + "\n\nOptions:\n", add_special_tokens=False), []
    for pos, ci in enumerate(order):
        ids += tok.encode("[%s] %s\n" % (LETTERS[pos], r["candidates"][ci]["text"]), add_special_tokens=False)
        ends.append(len(ids) - 1)
    ids += tok.encode("\nWhich option is the best next action? Answer with one letter.\nAnswer: [", add_special_tokens=False)
    return ids, ends, order

@torch.no_grad()
def capture(r):
    ids, ends, order = encode_pool(r)
    if len(ids) > CFG["max_tokens"]: return None     # dropped, counted by the caller
    x = torch.tensor([ids], device="cuda")
    o = model(input_ids=x, output_hidden_states=True, logits_to_keep=1)
    pos = torch.tensor(ends + [len(ids) - 1], device="cuda")
    hs = torch.stack([o.hidden_states[d][0, pos] for d in CFG["depths"]], 1).half().cpu()   # [n+1, 3, H]; hidden_states[d] = output of layer d
    n = len(order)
    probs = torch.softmax(o.logits[0, -1, LET_IDS[:n]].float(), -1).cpu()
    return dict(decision_id=r["decision_id"], order=order, gold_pos=order.index(0), q=hs[-1], cx=hs[:-1], qwen_probs=probs, n_tokens=len(ids))

t0, done, dropped, buf, shard = time.time(), 0, 0, [], 0
for i in range(0, len(rows), CFG["shard"]):
    fn = "%s/shard-%05d.pt" % (OUT, i // CFG["shard"])
    if os.path.exists(fn): continue          # resumable
    buf = []
    for r in rows[i:i + CFG["shard"]]:
        c = capture(r)
        if c is None: dropped += 1
        else: buf.append(c)
    torch.save(dict(protocol="pool_v1_kaggle", depths=CFG["depths"], qwen_revision=REV, items=buf), fn)
    done += len(buf); print(fn, len(buf), "items | %.1f s/decision | dropped %d" % ((time.time() - t0) / max(done, 1), dropped), flush=True)

In [ ]:
import glob, torch
items = [it for f in sorted(glob.glob(OUT + "/shard-*.pt")) for it in torch.load(f)["items"]]
top1 = sum(int(it["qwen_probs"].argmax()) == it["gold_pos"] for it in items) / max(len(items), 1)
print("decisions captured:", len(items), "| Qwen picks gold (options shown, no thinking):", round(top1, 3), "| chance ~", round(sum(1 / len(it["order"]) for it in items) / max(len(items), 1), 3))
json.dump(dict(label="exploratory (no gate)", protocol="pool_v1_kaggle", qwen_revision=REV, qwen_source=QWEN_SOURCE, cfg=CFG, decisions=len(items), qwen_gold_top1=top1, dropped_over_max_tokens=dropped), open(OUT + "/manifest.json", "w"), indent=1)